THIS NOTEBOOK IS FOR USAGE IN GOOGLE DRIVE (MODIFY DIRECTORY PATH AND STRUCTURE ACCORDING TO NEEDS BEFORE RUNNING)

In [ ]:
!pip install ultralytics grad-cam
import os
import shutil
import tarfile
import cv2
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image
from google.colab import drive

# ==========================================
# 0. COLAB & PATH CONFIGURATION
# ==========================================
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/My Drive/Railway'
TAR_PATH = os.path.join(DRIVE_DIR, 'dataset_split.tar')
LOCAL_BASE = '/content'
DATA_DIR = os.path.join(LOCAL_BASE, 'dataset_split')
MODELS_DIR = os.path.join(DRIVE_DIR, 'saved_best_models')

os.makedirs(MODELS_DIR, exist_ok=True)

# ==========================================
# 1. DATASET HANDLING
# ==========================================
def setup_local_data():
    if not os.path.exists(DATA_DIR):
        print(f"--- Extracting {TAR_PATH} ---")
        if os.path.exists(TAR_PATH):
            with tarfile.open(TAR_PATH, 'r') as tar_ref:
                tar_ref.extractall(LOCAL_BASE)
            print("Extraction complete!\n")
        else:
            raise FileNotFoundError(f"Missing {TAR_PATH}")

def organize_dataset(base_dir):
    splits = ['train', 'val', 'test']
    classes = ['rail', 'fastener', 'allgood']
    for split in splits:
        split_dir = os.path.join(base_dir, split)
        if not os.path.exists(split_dir): continue
        for cls in classes:
            os.makedirs(os.path.join(split_dir, cls), exist_ok=True)
        for filename in os.listdir(split_dir):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                filepath = os.path.join(split_dir, filename)
                if not os.path.isfile(filepath): continue
                for cls in classes:
                    if filename.startswith(cls):
                        shutil.move(filepath, os.path.join(split_dir, cls, filename))
                        break

# ==========================================
# 2. YOLOv8 GRAD-CAM WRAPPER
# ==========================================
class YOLOClsWrapper(nn.Module):
    """
    Wraps the YOLOv8 PyTorch model to return only the logits.
    Grad-CAM needs raw logits to backpropagate the gradients accurately.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x)
        # Ultralytics classification in eval mode returns a tuple: (probs, logits)
        if isinstance(out, (list, tuple)) and len(out) == 2:
            return out[1]
        return out

# ==========================================
# 3. MAIN EXECUTION BLOCK (FULL FIX)
# ==========================================
if __name__ == "__main__":
    # Prepare Data
    setup_local_data()
    organize_dataset(DATA_DIR)

    # ------------------------------------------
    # A. TRAIN YOLOv8
    # ------------------------------------------
    print(f"\n{'='*40}\n STARTING: YOLOv8-cls Training\n{'='*40}")

    yolo_model = YOLO('yolov8m-cls.pt')

    yolo_model.train(
        data=DATA_DIR,
        epochs=100,
        imgsz=224,
        batch=16,
        device=0,
        project=MODELS_DIR,
        name='YOLOv8_runs',
        patience=10,
        exist_ok=True
    )

    # ------------------------------------------
    # B. GRAD-CAM EXPLAINABILITY (FIXED)
    # ------------------------------------------
    print(f"\n{'='*40}\n STARTING: Grad-CAM Visualization\n{'='*40}")

    # 1. Setup Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 2. Load the best trained model
    best_model_path = os.path.join(MODELS_DIR, 'YOLOv8_runs', 'weights', 'bestrailer.pt')
    trained_yolo = YOLO(best_model_path)

    # 3. Wrap it and force it to the correct device
    wrapped_model = YOLOClsWrapper(trained_yolo.model)
    wrapped_model.to(device)
    wrapped_model.train() # Set to train mode to ensure gradients aren't suppressed

    # 4. Target the last convolutional layer
    target_layers = [trained_yolo.model.model[-2]]

    # 5. Initialize CAM
    cam = GradCAM(model=wrapped_model, target_layers=target_layers)

    # 6. Pick a sample test image
    sample_dir = os.path.join(DATA_DIR, 'test', 'fastener')
    images = [f for f in os.listdir(sample_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if not images:
        raise FileNotFoundError("No images found in test/fastener")

    sample_img_path = os.path.join(sample_dir, images[0])

    # 7. Preprocess image
    rgb_img = cv2.imread(sample_img_path)
    rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB)
    rgb_img = cv2.resize(rgb_img, (224, 224))
    rgb_img_float = np.float32(rgb_img) / 255.0

    # 8. Create input tensor & ENABLE GRADIENTS
    input_tensor = preprocess_image(rgb_img_float, mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0])
    input_tensor = input_tensor.to(device)
    input_tensor.requires_grad_(True) # Essential for Grad-CAM backprop

    # 9. Generate Heatmap with Gradient Enabled Context
    with torch.set_grad_enabled(True):
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]

    # 10. Overlay and Plot
    visualization = show_cam_on_image(rgb_img_float, grayscale_cam, use_rgb=True)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(rgb_img)
    plt.title(f"Original: {os.path.basename(sample_img_path)}")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(visualization)
    plt.title("Grad-CAM Heatmap")
    plt.axis('off')

    plt.tight_layout()
    plt.show()